# VIS4AI · Notebook 01 — Setup and first run

Goal of this notebook: get everything working and **let both models solve the same GSM8K math problems**.
No metrics or plots yet — just a clean setup and the raw answers, saved to Google Drive.

| Model | Role |
|---|---|
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | main model (a small *reasoning* model, thinks in `<think> … </think>`) |
| `Qwen/Qwen2.5-Math-1.5B` | baseline (same size, math-trained, no long reasoning) |

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (allow Google Drive access when asked)
3. **Disconnected?** Just *Run all* again. Answers are saved after every batch, and problems that are already done are skipped.

## 1. Setup
Install packages, check the GPU and connect Google Drive (where all results are stored).

In [ ]:
!pip install -q "transformers>=4.45" "accelerate>=0.33" "datasets>=2.20"

In [ ]:
import gc, json, math, os, re, time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

assert torch.cuda.is_available(), "No GPU! Go to Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Settings
All settings are in this one cell.
* `N_PROBLEMS`: how many GSM8K problems to try. Start small; raising it later only runs the new problems.
* `MAX_NEW_TOKENS`: maximum answer length. The reasoning model needs a lot (its thinking counts too).
* Sampling settings are DeepSeek's recommendation for this model (temperature 0.6, top-p 0.95).

In [ ]:
MODELS = {
    "r1-distill": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "qwen-math":  "Qwen/Qwen2.5-Math-1.5B",
}

N_PROBLEMS     = 50     # number of GSM8K test problems
MAX_NEW_TOKENS = 2048   # max tokens the model may write per problem
BATCH_SIZE     = 16     # problems generated at the same time (16 fits on a T4)
TEMPERATURE    = 0.6
TOP_P          = 0.95
SEED           = 42

RESULTS_DIR = Path("/content/drive/MyDrive/VIS4AI/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Results are saved in:", RESULTS_DIR)

## 2. Load the dataset
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) test split: 1,319 grade-school math word problems. The correct answer is the number after `####` in the solution text.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

problems = [
    {"id": i,
     "question": gsm8k[i]["question"],
     "gold": gsm8k[i]["answer"].split("####")[-1].strip().replace(",", "")}
    for i in range(N_PROBLEMS)
]

print(f"{len(problems)} problems loaded. Example:\n")
print(problems[0]["question"])
print("\nCorrect answer:", problems[0]["gold"])

## 3. Helper functions
* **`build_prompt`** — both models get the same prompt: the question plus *"Please reason step by step, and put your final answer within \boxed{}."* (DeepSeek's recommended format: no system prompt, everything in the user message). The tokenizer's chat template adds the model-specific special tokens.
* **`extract_answer`** — finds the last `\boxed{...}` in the output and turns it into a number.
* **`load_results` / `save_results`** — read/append a `.jsonl` file on Drive (one line per problem), which makes the run resumable.
* **`load_model` / `unload_model`** — load a model in float16 (the T4 has no bfloat16) and free the GPU afterwards.
* **`generate_answers`** — let the model answer a batch of questions.

In [ ]:
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

def build_prompt(tokenizer, question):
    messages = [{"role": "user", "content": question + "\n" + INSTRUCTION}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_answer(text):
    # Content of the last \boxed{...} (handles nested braces like \boxed{\frac{1}{2}}).
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    i, depth = start + len("\\boxed{"), 1
    for j in range(i, len(text)):
        depth += {"{": 1, "}": -1}.get(text[j], 0)
        if depth == 0:
            return text[i:j]
    return None   # output was cut off inside the box


def to_number(answer):
    # '$1,234' -> 1234.0, '20\%' -> 20.0, '\frac{3}{4}' -> 0.75, None if no number.
    if answer is None:
        return None
    a = re.sub(r"\\[dt]?frac\{(\d+)\}\{(\d+)\}", r"\1/\2", answer).replace(",", "")
    if m := re.fullmatch(r"\s*(-?\d+)/(\d+)\s*", a):
        return int(m.group(1)) / int(m.group(2))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", a)
    return float(numbers[-1]) if numbers else None


def is_correct(answer, gold):
    value = to_number(answer)
    return value is not None and math.isclose(value, float(gold), abs_tol=1e-6)


# Quick sanity check of the answer parsing
assert extract_answer("so \\boxed{18}.") == "18" and is_correct("18", "18")
assert is_correct(extract_answer("\\boxed{\\$1,234}"), "1234")
assert to_number(extract_answer("\\boxed{\\frac{3}{4}}")) == 0.75
assert extract_answer("no box here") is None and not is_correct(None, "5")
print("Answer parsing works.")

In [ ]:
def load_results(path):
    if not path.exists():
        return []
    results = []
    for line in path.read_text().splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            pass    # a half-written line from a disconnect is ignored (that problem is simply redone)
    return results


def save_results(path, results):
    with open(path, "a") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())    # make sure it really reaches Google Drive


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"           # needed for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda").eval()
    print(f"Loaded {model_name} | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, tokenizer


def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


@torch.no_grad()
def generate_answers(model, tokenizer, questions):
    prompts = [build_prompt(tokenizer, q) for q in questions]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    # Stop at the normal end token, and also at <|im_end|> (end of a chat turn for Qwen models)
    stop_ids = [tokenizer.eos_token_id]
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end != tokenizer.unk_token_id:
        stop_ids.append(im_end)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=stop_ids,
    )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [tokenizer.decode(t, skip_special_tokens=True) for t in new_tokens]


def run_model(model_key):
    model_name = MODELS[model_key]
    path = RESULTS_DIR / f"{model_key}.jsonl"
    done_ids = {r["id"] for r in load_results(path)}
    todo = [p for p in problems if p["id"] not in done_ids]
    print(f"{model_key}: {len(done_ids)} problems already done, {len(todo)} to go. Saving to {path}")
    if not todo:
        return

    model, tokenizer = load_model(model_name)
    set_seed(SEED)
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        t0 = time.time()
        outputs = generate_answers(model, tokenizer, [p["question"] for p in batch])
        results = []
        for p, out in zip(batch, outputs):
            answer = extract_answer(out)
            results.append({**p, "model": model_name, "output": out,
                            "answer": answer, "correct": is_correct(answer, p["gold"])})
        save_results(path, results)
        print(f"  saved {len(done_ids) + start + len(batch)}/{len(problems)}  ({time.time() - t0:.0f} s)")
    unload_model(model)

## 4. Run model 1: DeepSeek-R1-Distill-Qwen-1.5B
The reasoning model writes long "thinking" before answering, so one batch of 16 can take 1–2 minutes on a T4.

In [ ]:
run_model("r1-distill")

## 5. Run model 2: Qwen2.5-Math-1.5B
Same problems, same prompt. The first model was removed from the GPU, so there is room for this one.

In [ ]:
run_model("qwen-math")

## 6. Look at the answers
Both models side by side, one row per problem (read back from Drive, so this also works after a disconnect).

In [ ]:
tables = []
for key in MODELS:
    df = pd.DataFrame(load_results(RESULTS_DIR / f"{key}.jsonl"))
    if len(df):
        tables.append(df.drop_duplicates("id").set_index("id")[["answer", "correct"]].add_prefix(key + "_"))

comparison = pd.DataFrame(problems).set_index("id")[["gold"]].join(tables)
comparison

In [ ]:
# Read one full output. Change the model key or the problem id to look at others.
results = {r["id"]: r for r in load_results(RESULTS_DIR / "r1-distill.jsonl")}
example = results[problems[0]["id"]]
print("QUESTION:", example["question"])
print("\nCORRECT ANSWER:", example["gold"], "| MODEL ANSWER:", example["answer"])
print("\nMODEL OUTPUT:\n", example["output"])